In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import tune
from sklearn.utils import resample
from sklearn.preprocessing import OrdinalEncoder

import os
from sklearn.model_selection import cross_val_score
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv() 

import mlflow

mlflow.set_tracking_uri(os.getenv('MLFLOW_SERVER'))
mlflow.set_experiment("Kaggle-s6e5")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1778520156088, experiment_id='4', last_update_time=1778520156088, lifecycle_stage='active', name='Kaggle-s6e5', tags={}, trace_location=None, workspace='default'>

In [2]:
pipe, x, y = tune.process_data()

In [3]:
model_name_hgb = "Best_HGB_s6ep5"
model_name_lgbm = "Best_LGBM_s6ep5"
model_name_cb = "Best_CB_s6ep5"
model_version = "latest"

# # Load the model from the Model Registry
model_uri_hgb = f"models:/{model_name_hgb}/{model_version}"
model_uri_lgbm = f"models:/{model_name_lgbm}/{model_version}"
model_uri_cb = f"models:/{model_name_cb}/{model_version}"

model_hgb = mlflow.sklearn.load_model(model_uri_hgb)
model_lgbm = mlflow.sklearn.load_model(model_uri_lgbm)
model_cb = mlflow.sklearn.load_model(model_uri_cb)

# model.fit(x, y)

In [4]:
from sklearn.pipeline import Pipeline

p1 = Pipeline(steps=[('preprocessor', pipe), ('model', model_hgb)])
p2 = Pipeline(steps=[('preprocessor', pipe), ('model', model_lgbm)])
p3 = Pipeline(steps=[('preprocessor', pipe), ('model', model_cb)])

In [5]:
from my_sklearn_nn import MyNNClassifier
import tune
# pipe, x, y = tune.process_data()
my_nn_clf = MyNNClassifier()
#pipe.steps.append(['classifier_nn', my_nn_clf])
#pipe.fit(x, y)
p4 = Pipeline(steps=[('preprocessor', pipe), ('model', my_nn_clf)])

In [6]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[('hgb', p1), ('lgbm', p2), ('cb', p3), ('nn', p4)], voting='soft')
#cv_scores = cross_val_score(voting_clf, x, y, cv=2, scoring='balanced_accuracy', error_score='raise')
#print(f"Mean cross-validation score: {np.mean(cv_scores):.3f}")
voting_clf.fit(x, y)

[LightGBM] [Warning] min_data_in_leaf is set=19, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=19
[LightGBM] [Warning] min_data_in_leaf is set=19, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=19
[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002699 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1245
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warni

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('hgb', ...), ('lgbm', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier

base_models = [('hgb', p1), ('lgbm', p2), ('cb', p3), ('nn', p4)]
meta_model = LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=1)
#meta_model = DecisionTreeClassifier(max_depth=3, random_state=1)
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_clf.fit(x, y)

In [ ]:
with mlflow.start_run(nested=False, run_name=f"trial_{model_name_hgb}_{model_name_lgbm}_{model_name_cb}_stacking"):
    # mlflow.log_param("model_hgb", model_name_hgb)
    # mlflow.log_param("model_lgbm", model_name_lgbm)
    # mlflow.log_param("model_cb", model_name_cb)
    # mlflow.log_param("meta_model", "LogisticRegression")
    mlflow.log_param("params", stacking_clf.get_params(deep=True))
    #mlflow.sklearn.log_model(voting_clf, "voting_clf")
    mlflow.sklearn.log_model(stacking_clf, "stacking_clf")
    mlflow.log_metric("cv_score", np.mean(cross_val_score(stacking_clf, x, y, cv=5, scoring='roc_auc', error_score='raise')))

In [ ]:
testing_data = pd.read_csv("./data/test.csv")
#preds_voting = voting_clf.predict_proba(testing_data)[:, 1]
preds_stacking = stacking_clf.predict_proba(testing_data)[:, 1]



In [ ]:
submission = pd.DataFrame({
    "id": testing_data["id"],
    "PitNextLap": preds_voting
})

out_dir = 'data/'

submission.to_csv(os.path.join(out_dir, 'voting_clf_tuned_v1.csv'), index=False)

In [ ]:
submission_stacking = pd.DataFrame({
    "id": testing_data["id"],
    "PitNextLap": preds_stacking
})

out_dir = 'data/'

submission_stacking.to_csv(os.path.join(out_dir, 'stacking_clf_tuned_v2.csv'), index=False)